Search Engine with Tools and Agents
This code also creates a custom tool using the RAG pattern.

In [34]:
## Arxiv & Wikipedia Research
# Tool Creation
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

In [35]:
## Used the inbuilt tool of wikipedia
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [36]:
## Used the inbuilt tool of Arxiv
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [37]:
# Combine the tools
tools = [wiki,arxiv]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/Users/satyakibasu/Documents/Satyaki/python_code/gen-ai/my-langchain/p310env/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250))]

Create you own Custom Tool

In [38]:
## Custom Tool using RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
from dotenv import load_dotenv

load_dotenv()

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["GITHUB_TOKEN"],  # Your GitHub PAT
    base_url="https://models.inference.ai.azure.com")

In [39]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load() #--> Gives all the Documents objects
documents = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=50).split_documents(docs)
print(f"Total chunks created: {len(documents)}")

Total chunks created: 54


In [40]:
vectordb = FAISS.from_documents(documents=documents, embedding=embedding_model)
retriever = vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ff153544d30>, search_kwargs={})

In [41]:
## Convert the Retriever into a Tool
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(retriever=retriever, name="langsmith-search",description="Search any information on LangSmith")
retriever_tool.name

'langsmith-search'

In [42]:
tools.append(retriever_tool)
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/Users/satyakibasu/Documents/Satyaki/python_code/gen-ai/my-langchain/p310env/lib/python3.10/site-packages/wikipedia/__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith-search', description='Search any information on LangSmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x7ff120416b90>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x7ff1204167a0>)]

Run all the tools using LLM and agents

In [43]:
from langchain_groq import ChatGroq # --> will use GROQ Infra for accessing LLM models 

# Load the GROQ key and create the LLM model
groq_api_key = os.getenv('GROQ_API_KEY')
llm = ChatGroq(groq_api_key=groq_api_key,model='llama-3.1-8b-instant')

In [ ]:
## Create and execute Agents
# Note - During agents, we do not create a ChatPromptTemplate. 
"""
ChatPromptTemplate is now meant for:
Chains
RAG pipelines
Custom runnables
Non-agent workflows
"""

from langchain.agents import create_agent
from rich import print

agent = create_agent(
    model=llm,
    tools=tools, 
    system_prompt="You are an helpful assistant." 
)

response = agent.invoke({
                        "messages":
                                {"role": "user", "content":"Tell me about Transformer"}
                         }
)


from langchain_core.messages import AIMessage, ToolMessage

print(response['messages'])

# This code scans the response and give which tool calls are being made
for msg in response["messages"]:
    if isinstance(msg, AIMessage) and msg.tool_calls:
        print("Tool call detected:", msg.tool_calls)

print(response['messages'][-1].content)



/Users/satyakibasu/Documents/Satyaki/python_code/gen-ai/my-langchain/p310env/lib/python3.10/site-packages/langchain_community/utilities/arxiv.py:105: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  ).results()


[
    HumanMessage(
        content='Tell me about Transformer',
        additional_kwargs={},
        response_metadata={},
        id='ae35a83e-faec-43fb-9ed9-1a872ff5a74a'
    ),
    AIMessage(
        content='',
        additional_kwargs={
            'tool_calls': [
                {
                    'id': 'v6qf3t6z6',
                    'function': {'arguments': '{"query":"Transformer (machine learning)"}', 'name': 'arxiv'},
                    'type': 'function'
                }
            ]
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 18,
                'prompt_tokens': 503,
                'total_tokens': 521,
                'completion_time': 0.052734817,
                'completion_tokens_details': None,
                'prompt_time': 0.056498288,
                'prompt_tokens_details': None,
                'queue_time': 0.056658672,
                'total_time': 0.109233105
            },
            'model_name': 'llama-3.1-8b-instant',
            'system_fingerprint': 'fp_ff2b098aaf',
            'service_tier': 'on_demand',
            'finish_reason': 'tool_calls',
            'logprobs': None,
            'model_provider': 'groq'
        },
        id='lc_run--019b82a7-82e2-79f3-a2e2-0b51ea0135dd-0',
        tool_calls=[
            {
                'name': 'arxiv',
                'args': {'query': 'Transformer (machine learning)'},
                'id': 'v6qf3t6z6',
                'type': 'tool_call'
            }
        ],
        usage_metadata={'input_tokens': 503, 'output_tokens': 18, 'total_tokens': 521}
    ),
    ToolMessage(
        content='Published: 2023-06-07\nTitle: Changing Data Sources in the Age of Machine Learning for Official 
Statistics\nAuthors: Cedric De Boom, Michael Reusens\nSummary: Data science has become increasingly essential for 
the production of official statistics, as i',
        name='arxiv',
        id='b7ec0a3d-68f9-4e5f-8ba2-0c580f8ab194',
        tool_call_id='v6qf3t6z6'
    ),
    AIMessage(
        content='',
        additional_kwargs={
            'tool_calls': [
                {
                    'id': 'ex5sht3sx',
                    'function': {
                        'arguments': '{"query":"Transformer (machine learning)"}',
                        'name': 'langsmith-search'
                    },
                    'type': 'function'
                }
            ]
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 19,
                'prompt_tokens': 585,
                'total_tokens': 604,
                'completion_time': 0.020565275,
                'completion_tokens_details': None,
                'prompt_time': 0.033084557,
                'prompt_tokens_details': None,
                'queue_time': 0.055832743,
                'total_time': 0.053649832
            },
            'model_name': 'llama-3.1-8b-instant',
            'system_fingerprint': 'fp_4387d3edbb',
            'service_tier': 'on_demand',
            'finish_reason': 'tool_calls',
            'logprobs': None,
            'model_provider': 'groq'
        },
        id='lc_run--019b82a7-8495-7011-8677-87d6472de89c-0',
        tool_calls=[
            {
                'name': 'langsmith-search',
                'args': {'query': 'Transformer (machine learning)'},
                'id': 'ex5sht3sx',
                'type': 'tool_call'
            }
        ],
        usage_metadata={'input_tokens': 585, 'output_tokens': 19, 'total_tokens': 604}
    ),
    ToolMessage(
        content='monitoring and evaluation to build more reliable AI systems.\n\nand deploy AI agents visually with
a no-code interface—perfect for rapid prototyping and getting\n\ntracingEvaluationMeasure and track quality over 
time to ensure your AI applications are consistent\n\nto set up LangSmithAgent Builder (Beta)Design and deploy AI 
agents visually with a no-code',
        name='lang

Tool call detected:
[{'name': 'arxiv', 'args': {'query': 'Transformer (machine learning)'}, 'id': 'v6qf3t6z6', 'type': 'tool_call'}]

Tool call detected:
[
    {
        'name': 'langsmith-search',
        'args': {'query': 'Transformer (machine learning)'},
        'id': 'ex5sht3sx',
        'type': 'tool_call'
    }
]

Tool call detected:
[
    {
        'name': 'wikipedia',
        'args': {'query': 'Transformer (machine learning)'},
        'id': '1r2qj2kbr',
        'type': 'tool_call'
    }
]

The Transformer architecture is a type of neural network that has gained popularity in recent years due to its 
performance on various natural language processing tasks, such as machine translation, text summarization, and 
question answering. It is based on the self-attention mechanism, which allows the model to weigh the importance of 
different input elements relative to each other, rather than relying on a fixed positional encoding. This makes the
Transformer architecture particularly well-suited for tasks that involve long-range dependencies in the input data.

The Transformer architecture consists of an encoder and a decoder. The encoder takes in a sequence of tokens and 
outputs a continuous representation of the input sequence. The decoder takes this representation and generates an 
output sequence, one token at a time.

One of the key advantages of the Transformer architecture is its ability to handle long-range dependencies in the 
input data. This is because the self-attention mechanism allows the model to weigh the importance of different 
input elements relative to each other, rather than relying on a fixed positional encoding. This makes the 
Transformer architecture particularly well-suited for tasks that involve long-range dependencies in the input data,
such as machine translation and text summarization.

Another advantage of the Transformer architecture is its ability to perform well on tasks that involve multiple 
input sequences. This is because the self-attention mechanism allows the model to weigh the importance of different
input elements relative to each other, rather than relying on a fixed positional encoding. This makes the 
Transformer architecture particularly well-suited for tasks that involve multiple input sequences, such as question
answering and dialogue systems.

Overall, the Transformer architecture is a powerful tool for natural language processing tasks, and its ability to 
handle long-range dependencies and perform well on tasks that involve multiple input sequences makes it a popular 
choice for many applications.